<a href="https://colab.research.google.com/github/keyvanmahmoudi/Data_Science_Course/blob/main/sessions/Session03_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۳: پاک‌سازی داده با Pandas

یک ضرب‌المثل قدیمی در علم داده: *Garbage in, garbage out*. اگر داده کثیف باشد، دقیق‌ترین تحلیل هم نتیجهٔ غلط می‌دهد.

### اهداف این جلسه

- شناسایی مشکلات رایج دادهٔ واقعی: تکرار، نوع نادرست، متن ناسازگار، مقدار گمشده، مقدار غیرممکن
- تبدیل نوع ستون‌ها (متن به عدد، متن به تاریخ)
- یکسان‌سازی متن با متدهای `.str`
- راهبردهای برخورد با مقادیر گمشده
- شناسایی دادهٔ پرت با روش IQR
- نوشتن **بررسی‌های اعتبارسنجی** تا مطمئن شویم داده واقعاً تمیز شده است

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. مجموعه‌داده: سفارش‌های یک فروشگاه اینترنتی

این داده **ساختگی** است و برای آموزش طراحی شده، اما مشکلاتش دقیقاً همان‌هایی است که در داده‌های واقعی می‌بینیم: سفارش‌های یک فروشگاه اینترنتی در یک سال (از ۱ فروردین ۱۴۰۳). قیمت‌ها به تومان است.

| ستون | توضیح |
|---|---|
| `order_id` | شناسهٔ سفارش |
| `order_date` | تاریخ و ساعت ثبت سفارش |
| `customer_id`, `product_id` | شناسهٔ مشتری و کالا |
| `city` | شهر مشتری |
| `quantity` | تعداد |
| `total_price` | مبلغ کل (تومان) |
| `payment_method` | روش پرداخت |
| `delivery_days` | مدت ارسال (روز) |
| `rating` | امتیاز مشتری (۱ تا ۵) |

</div>

In [1]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/keyvanmahmoudi/Data_Science_Course/main/data/"

raw = pd.read_csv(DATA_URL + "orders_raw.csv")
df = raw.copy()          # always keep the raw data untouched
print(df.shape)
df.head()

(1218, 10)


,order_id,order_date,customer_id,city,product_id,quantity,total_price,payment_method,delivery_days,rating
0,100001,2024-03-20 13:31,C0103,Shiraz,P007,2,"720,000",card,4,5.0
1,100002,2024-03-20 13:45,C0151,Rasht,P001,1,2400000,Card,4,4.0
2,100003,2024-03-20 16:13,C0132,Amol,P006,2,840000,cash_on_delivery,3,3.0
3,100004,2024-03-20 18:16,C0142,Tehran,P002,4,4600000,card,7,4.0
4,100005,2024-03-20 19:37,C0082,Mashhad,P005,2,"6,080,000",card,4,5.0


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        1218 non-null   int64  
 1   order_date      1218 non-null   str    
 2   customer_id     1218 non-null   str    
 3   city            1191 non-null   str    
 4   product_id      1218 non-null   str    
 5   quantity        1218 non-null   int64  
 6   total_price     1218 non-null   str    
 7   payment_method  1206 non-null   str    
 8   delivery_days   1218 non-null   int64  
 9   rating          1076 non-null   float64
dtypes: float64(1), int64(3), str(6)
memory usage: 95.3 KB


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

همین خروجی چند زنگ خطر دارد:

- `total_price` باید عدد باشد، اما نوعش متنی است.
- `order_date` باید تاریخ باشد، اما متن است.
- `city`، `payment_method` و `rating` مقدار گمشده دارند.

بیایید یکی‌یکی و با ترتیب درست سراغشان برویم.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. سطرهای تکراری

گاهی یک سفارش دو بار ثبت می‌شود (مثلاً مشتری دو بار روی دکمهٔ پرداخت کلیک کرده است). اول تکرارهای **کامل** را بررسی می‌کنیم و بعد تکرار **کلید** (`order_id` باید یکتا باشد).

</div>

In [3]:
print("fully duplicated rows:", df.duplicated().sum())
print("duplicated order_id  :", df["order_id"].duplicated().sum())
df[df["order_id"].duplicated(keep=False)].sort_values("order_id").head(4)

fully duplicated rows: 18
duplicated order_id  : 18


,order_id,order_date,customer_id,city,product_id,quantity,total_price,payment_method,delivery_days,rating
34,100035,2024-03-31 14:40,C0058,Tabriz,P008,2,"560,000",wallet,15,4.0
35,100035,2024-03-31 14:40,C0058,Tabriz,P008,2,"560,000",wallet,15,4.0
47,100047,2024-04-03 18:17,C0150,Isfahan,P009,1,510000,card,2,3.0
48,100047,2024-04-03 18:17,C0150,Isfahan,P009,1,510000,card,2,3.0


In [4]:
df = df.drop_duplicates()
assert df["order_id"].is_unique
print(df.shape)

(1200, 10)


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

> **چرا اول تکرارها؟** اگر قبل از حذف تکرارها میانگین یا مقدار گمشده را حساب کنیم، سطرهای تکراری وزن بیشتری می‌گیرند و نتیجه را منحرف می‌کنند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۳. تبدیل نوع: متن به عدد

چرا `total_price` متنی است؟ بیایید مقادیری را ببینیم که به عدد تبدیل نمی‌شوند. `pd.to_numeric` با `errors="coerce"` هر مقدار غیرقابل‌تبدیل را `NaN` می‌کند و این کمک می‌کند مقادیر مشکل‌دار را پیدا کنیم:

</div>

In [5]:
as_number = pd.to_numeric(df["total_price"], errors="coerce")
bad = df.loc[as_number.isna(), "total_price"]
print(len(bad), "values could not be parsed, e.g.:")
print(bad.head().tolist())

260 values could not be parsed, e.g.:
['720,000', '6,080,000', '4,560,000', '520,000', '3,800,000']


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

مشکل جداکنندهٔ هزارگان (`,`) است. آن را حذف می‌کنیم و بعد تبدیل می‌کنیم. این بار **بدون** `coerce`، تا اگر مشکل دیگری بود خطا بگیریم و متوجه شویم:

</div>

In [6]:
df["total_price"] = df["total_price"].astype(str).str.replace(",", "", regex=False).astype("int64")
df["total_price"].describe().round(0)

count        1200.0
mean      2667768.0
std       3625659.0
min        144000.0
25%        560000.0
50%       1300000.0
75%       3095000.0
max      27200000.0
Name: total_price, dtype: float64

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۴. تبدیل نوع: متن به تاریخ

با تبدیل به نوع `datetime` می‌توانیم سال، ماه، روز هفته و ساعت را استخراج کنیم و فاصلهٔ زمانی حساب کنیم. دادن `format` صریح هم سریع‌تر است و هم جلوی تفسیر اشتباه را می‌گیرد.

</div>

In [7]:
df["order_date"] = pd.to_datetime(df["order_date"], format="%Y-%m-%d %H:%M")
print(df["order_date"].min(), "->", df["order_date"].max())

df["month"] = df["order_date"].dt.to_period("M")
df["weekday"] = df["order_date"].dt.day_name()
df["hour"] = df["order_date"].dt.hour
df[["order_date", "month", "weekday", "hour"]].head(3)

2024-03-20 13:31:00 -> 2025-03-19 22:16:00


,order_date,month,weekday,hour
0,2024-03-20 13:31:00,2024-03,Wednesday,13
1,2024-03-20 13:45:00,2024-03,Wednesday,13
2,2024-03-20 16:13:00,2024-03,Wednesday,16


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۵. یکسان‌سازی متن

ستون `city` را ببینیم:

</div>

In [8]:
df["city"].value_counts(dropna=False)

city
Tehran     304
Shiraz     148
Mashhad    130
Tabriz     118
Isfahan    101
Rasht       95
Sari        94
Amol        40
tehran      26
NaN         25
sari        20
mashhad     17
TEHRAN      16
Esfahan     14
 Tehran     13
Tehran      12
Mashad      11
amol         9
isfahan      7
Name: count, dtype: int64

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

«Tehran»، «tehran»، «TEHRAN» و « Tehran» برای انسان یکی‌اند، اما برای کامپیوتر چهار مقدار متفاوت. اگر الان بر اساس شهر گروه‌بندی کنیم، فروش تهران به چند تکه تقسیم می‌شود. راه‌حل دو مرحله دارد:

1. حذف فاصله‌های اضافه و یکسان کردن حروف (`str.strip`، `str.title`)
2. یکی کردن املاهای مختلف با یک دیکشنری

</div>

In [9]:
df["city"] = df["city"].str.strip().str.title()
df["city"] = df["city"].replace({"Esfahan": "Isfahan", "Mashad": "Mashhad"})
df["city"].value_counts(dropna=False)

city
Tehran     371
Mashhad    158
Shiraz     148
Isfahan    122
Tabriz     118
Sari       114
Rasht       95
Amol        49
NaN         25
Name: count, dtype: int64

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

همین کار را برای روش پرداخت انجام می‌دهیم:

</div>

In [10]:
print(df["payment_method"].value_counts(dropna=False), "\n")
df["payment_method"] = (
    df["payment_method"].str.strip().str.lower()
      .replace({"cod": "cash_on_delivery"})
)
df["payment_method"].value_counts(dropna=False)

payment_method
card                688
wallet              275
cash_on_delivery    170
Card                 40
COD                  15
NaN                  12
Name: count, dtype: int64 



payment_method
card                728
wallet              275
cash_on_delivery    185
NaN                  12
Name: count, dtype: int64

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۶. مقادیر غیرممکن

بعضی مقادیر از نظر نوع درست‌اند اما از نظر **منطق کسب‌وکار** غیرممکن‌اند. برای پیدا کردنشان باید قواعد دامنه را بدانیم:

- تعداد کالا نمی‌تواند منفی باشد.
- امتیاز باید بین ۱ تا ۵ باشد.
- ارسال ۱۲۰ روزه برای یک فروشگاه داخلی عملاً غیرممکن است.

</div>

In [11]:
print("quantity <= 0     :", (df["quantity"] <= 0).sum())
print("rating out of 1..5:", (~df["rating"].between(1, 5) & df["rating"].notna()).sum())
df["delivery_days"].describe()

quantity <= 0     : 4
rating out of 1..5: 2


count    1200.000000
mean        4.415833
std         6.147309
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max       120.000000
Name: delivery_days, dtype: float64

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### شناسایی دادهٔ پرت با روش IQR

یک قاعدهٔ رایج و مقاوم: مقادیر بیرون از بازهٔ زیر «پرت» در نظر گرفته می‌شوند

$$[Q_1 - 1.5 \times IQR,\; Q_3 + 1.5 \times IQR], \qquad IQR = Q_3 - Q_1$$

که $Q_1$ و $Q_3$ چارک اول و سوم هستند. این روش برخلاف روش مبتنی بر میانگین و انحراف معیار، خودش تحت تأثیر دادهٔ پرت قرار نمی‌گیرد.

</div>

In [12]:
q1, q3 = df["delivery_days"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr}, fence=[{low}, {high}]")
outliers = df[(df["delivery_days"] < low) | (df["delivery_days"] > high)]
outliers["delivery_days"].value_counts().sort_index()

Q1=3.0, Q3=5.0, IQR=2.0, fence=[0.0, 8.0]


delivery_days
9      15
10     13
11      5
12      6
13      2
15      1
120     3
Name: count, dtype: int64

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**نکتهٔ مهم:** «پرت» با «غلط» یکی نیست. ارسال ۹ تا ۱۵ روزه طبق قاعدهٔ IQR پرت است، اما کاملاً ممکن است (مثلاً ارسال به شهر دور یا تعطیلات نوروز). این‌ها دادهٔ **واقعی** هستند و نباید حذف شوند. فقط مقدار ۱۲۰ روز غیرممکن است.

پس تصمیم ما:
- مقادیر غیرممکن را به `NaN` تبدیل می‌کنیم (اطلاعات بقیهٔ ستون‌های آن سفارش حفظ شود). متد `where` مقادیری را که شرط برایشان برقرار **نیست** و `mask` مقادیری را که شرط برایشان برقرار **هست**، با `NaN` جایگزین می‌کند.
- سفارش با تعداد منفی را حذف می‌کنیم، چون مبلغ آن هم قابل‌اعتماد نیست.

</div>

In [13]:
df["rating"] = df["rating"].where(df["rating"].between(1, 5))
df["delivery_days"] = df["delivery_days"].mask(df["delivery_days"] > 60)
df = df[df["quantity"] > 0].copy()
print(df.shape)

(1196, 13)


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۷. مقادیر گمشده

قبل از پر کردن مقادیر گمشده باید بپرسیم: **چرا** گمشده‌اند؟

- `rating`: بسیاری از مشتری‌ها اصلاً امتیاز نمی‌دهند. گمشده بودن خودش یک اطلاع است («امتیاز نداده»). پر کردنش با میانگین، این واقعیت را پنهان می‌کند و توزیع امتیازها را مصنوعی می‌کند. **تصمیم:** پر نمی‌کنیم و یک ستون پرچم می‌سازیم.
- `city`: شهر هر مشتری در جدول مشتری‌ها هم هست؛ می‌توانیم از سفارش‌های دیگر **همان مشتری** پرش کنیم.
- `payment_method`: تعداد کم است؛ با مقدار `"unknown"` پر می‌کنیم تا گم نشوند.
- `delivery_days`: چند مقدار معدود؛ با **میانه** پر می‌کنیم که به دادهٔ پرت حساس نیست.

</div>

In [14]:
df.isna().sum()[lambda s: s > 0]

city               25
payment_method     12
delivery_days       3
rating            142
dtype: int64

In [15]:
df["has_rating"] = df["rating"].notna()

city_by_customer = df.dropna(subset=["city"]).groupby("customer_id")["city"].agg(lambda s: s.mode().iloc[0])
df["city"] = df["city"].fillna(df["customer_id"].map(city_by_customer))

df["payment_method"] = df["payment_method"].fillna("unknown")
df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median()).astype(int)

df.isna().sum()[lambda s: s > 0]

rating    142
dtype: int64

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

فقط `rating` گمشده مانده که تصمیم آگاهانهٔ ما بود. (`groupby` را در جلسهٔ بعد مفصل یاد می‌گیریم.)

## ۸. نوع دسته‌ای (category)

ستون‌هایی که تعداد کمی مقدار تکراری دارند را می‌توان به نوع `category` تبدیل کرد. این کار حافظه را کم می‌کند و جلوی مقادیر نامعتبر را هم می‌گیرد.

</div>

In [16]:
before = df.memory_usage(deep=True).sum()
for col in ["city", "payment_method", "weekday"]:
    df[col] = df[col].astype("category")
after = df.memory_usage(deep=True).sum()
print(f"memory: {before / 1024:.0f} KB -> {after / 1024:.0f} KB")

memory: 400 KB -> 210 KB


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۹. اعتبارسنجی نهایی

پاک‌سازی بدون بررسی نهایی کامل نیست. با `assert` قواعدی را که دادهٔ تمیز **باید** رعایت کند، صریحاً می‌نویسیم. اگر روزی دادهٔ جدیدی با همین کد پاک شود و یکی از قواعد نقض شود، فوراً خطا می‌گیریم.

</div>

In [17]:
assert df["order_id"].is_unique
assert (df["quantity"] > 0).all()
assert df["rating"].dropna().between(1, 5).all()
assert df["delivery_days"].between(1, 60).all()
assert df["city"].notna().all()
assert set(df["payment_method"].cat.categories) <= {"card", "wallet", "cash_on_delivery", "unknown"}
assert pd.api.types.is_datetime64_any_dtype(df["order_date"])
print("All checks passed:", df.shape)

All checks passed: (1196, 14)


In [18]:
summary = pd.DataFrame({
    "raw": [len(raw), raw["city"].nunique(), raw["payment_method"].nunique(), str(raw["total_price"].dtype)],
    "clean": [len(df), df["city"].nunique(), df["payment_method"].nunique(), str(df["total_price"].dtype)],
}, index=["rows", "distinct cities", "distinct payment labels", "total_price dtype"])
summary

,raw,clean
rows,1218,1196
distinct cities,18,8
distinct payment labels,5,4
total_price dtype,str,int64


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱۰. همه‌چیز در یک تابع

در یک پروژهٔ واقعی، مراحل پاک‌سازی را در یک تابع جمع می‌کنیم تا روی هر نسخهٔ جدید داده دوباره قابل اجرا باشد. خروجی همین تابع با نام `orders_clean.csv` در پوشهٔ `data` مخزن هم قرار دارد و در جلسه‌های بعد از آن استفاده می‌کنیم.

</div>

In [19]:
CITY_FIX = {"Esfahan": "Isfahan", "Mashad": "Mashhad"}

def clean_orders(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.drop_duplicates().copy()
    df["total_price"] = df["total_price"].astype(str).str.replace(",", "", regex=False).astype("int64")
    df["order_date"] = pd.to_datetime(df["order_date"], format="%Y-%m-%d %H:%M")
    df["city"] = df["city"].str.strip().str.title().replace(CITY_FIX)
    df["payment_method"] = (df["payment_method"].str.strip().str.lower()
                            .replace({"cod": "cash_on_delivery"}).fillna("unknown"))
    df = df[df["quantity"] > 0].copy()
    df["rating"] = df["rating"].where(df["rating"].between(1, 5))
    df["delivery_days"] = df["delivery_days"].mask(df["delivery_days"] > 60)
    df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median()).astype(int)
    known_city = df.dropna(subset=["city"]).groupby("customer_id")["city"].agg(lambda s: s.mode().iloc[0])
    df["city"] = df["city"].fillna(df["customer_id"].map(known_city))
    return df.reset_index(drop=True)

orders = clean_orders(raw)
orders.to_csv("orders_clean.csv", index=False)
print(orders.shape)

(1196, 10)


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## جمع‌بندی: یک چک‌لیست پاک‌سازی

1. از دادهٔ خام کپی بگیرید و آن را دست نزنید.
2. تکرارها را اول حذف کنید و یکتایی کلید را بررسی کنید.
3. نوع هر ستون را درست کنید (عدد، تاریخ، دسته).
4. متن‌ها را یکسان کنید (فاصله، حروف، املا).
5. مقادیر غیرممکن را با دانش دامنه پیدا کنید؛ پرت را با غلط اشتباه نگیرید.
6. برای هر ستون گمشده، **دلیل** را بفهمید و بعد راهبرد انتخاب کنید.
7. قواعد نهایی را با `assert` بنویسید.
8. کل فرایند را در یک تابع تکرارپذیر بنویسید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. در `raw`، چند سفارش مبلغی دارند که بر قیمت واحد کالا (از فایل `products.csv`) بخش‌پذیر نیست؟ این یعنی چه؟ (راهنما: تخفیف)

۲. ستونی به نام `is_weekend` بسازید که برای سفارش‌های پنجشنبه و جمعه `True` باشد.

۳. به‌جای میانه، `delivery_days` گمشده را با میانهٔ **همان شهر** پر کنید. (راهنما: `groupby(...).transform("median")`)

۴. با روش IQR، سفارش‌های پرت از نظر `total_price` را پیدا کنید. آیا به نظرتان غلط هستند یا واقعی؟

۵. تابع `clean_orders` را طوری تغییر دهید که گزارشی از تعداد سطرهای حذف‌شده در هر مرحله چاپ کند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** در `raw`، چند سفارش مبلغی دارند که بر قیمت واحد کالا (از فایل `products.csv`) بخش‌پذیر نیست؟ این یعنی چه؟ (راهنما: تخفیف)

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** ستونی به نام `is_weekend` بسازید که برای سفارش‌های پنجشنبه و جمعه `True` باشد.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** به‌جای میانه، `delivery_days` گمشده را با میانهٔ **همان شهر** پر کنید. (راهنما: `groupby(...).transform("median")`)

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** با روش IQR، سفارش‌های پرت از نظر `total_price` را پیدا کنید. آیا به نظرتان غلط هستند یا واقعی؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید


<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۵.** تابع `clean_orders` را طوری تغییر دهید که گزارشی از تعداد سطرهای حذف‌شده در هر مرحله چاپ کند.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید
